In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
    RocCurveDisplay,
)
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

sns.set_theme(style="whitegrid")

In [ ]:
data_path = Path("diabetes.csv")
if not data_path.is_file():
    data_path = Path("Assignment_1") / "diabetes.csv"
if not data_path.is_file():
    raise FileNotFoundError("Could not find diabetes.csv beside the notebook or in Assignment_1.")

df = pd.read_csv(data_path)
df.head()

In [ ]:
print("Dataset shape:", df.shape)
print("\nColumn names:")
print(df.columns.tolist())
print("\nData types:")
print(df.dtypes)
print("\nMissing values:")
print(df.isna().sum())
print("\nStatistical summary:")
display(df.describe())

In [ ]:
plt.figure(figsize=(6, 4))
sns.countplot(data=df, x="Outcome")
plt.title("Diabetes Outcome Distribution")
plt.xlabel("Outcome (0 = no diabetes, 1 = diabetes)")
plt.ylabel("Number of records")
plt.show()

print("Outcome counts:")
print(df["Outcome"].value_counts().sort_index())

In [ ]:
target_column = "Outcome"
zero_as_missing = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]

X = df.drop(columns=target_column).copy()
y = df[target_column].copy()

for column in zero_as_missing:
    X[column] = X[column].astype(float).replace(0, np.nan)

print("Feature shape:", X.shape)
print("Target shape:", y.shape)
print("\nMissing values after treating invalid zeros as missing:")
print(X.isna().sum())

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

print("Training data:", X_train.shape, "class counts:", y_train.value_counts().sort_index().to_dict())
print("Testing data: ", X_test.shape, "class counts:", y_test.value_counts().sort_index().to_dict())

In [ ]:
model = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("classifier", LogisticRegression(max_iter=1000, random_state=42)),
    ]
)

model.fit(X_train, y_train)
print("Model trained successfully.")

y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)
train_accuracy = accuracy_score(y_train, y_train_pred)
test_accuracy = accuracy_score(y_test, y_test_pred)

print(f"Training accuracy: {train_accuracy:.3f}")
print(f"Testing accuracy:  {test_accuracy:.3f}")

In [ ]:
print(f"Precision (diabetes): {precision_score(y_test, y_test_pred):.3f}")
print(f"Recall (diabetes):    {recall_score(y_test, y_test_pred):.3f}")
print(f"F1-score (diabetes):  {f1_score(y_test, y_test_pred):.3f}")
print("\nClassification report:")
print(classification_report(y_test, y_test_pred, target_names=["No diabetes", "Diabetes"]))

cm = confusion_matrix(y_test, y_test_pred, labels=[0, 1])
plt.figure(figsize=(5, 4))
sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues",
    xticklabels=["No diabetes", "Diabetes"],
    yticklabels=["No diabetes", "Diabetes"],
)
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.title("Test Confusion Matrix")
plt.show()

In [ ]:
test_probabilities = model.predict_proba(X_test)[:, 1]
print(f"Test ROC-AUC: {roc_auc_score(y_test, test_probabilities):.3f}")

RocCurveDisplay.from_estimator(model, X_test, y_test)
plt.title("ROC Curve — Test Set")
plt.show()